In [11]:
import sys
sys.path.append("..")


In [ ]:

import shared_libraries.data_processing_utils as processing
import pandas as pd
import pickle

from typing import Dict, List

In [4]:
with open("../ig_sessions.pickle", "rb") as file:
    sessions = pickle.load(file)

In [28]:
compounds_map = pd.read_csv("../ig_compounds_map.csv")

In [30]:
sessions[0].session_info

{'Meeting': {'Key': 1124,
  'Name': 'Bahrain Grand Prix',
  'OfficialName': 'FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022',
  'Location': 'Sakhir',
  'Country': {'Key': 36, 'Code': 'BRN', 'Name': 'Bahrain'},
  'Circuit': {'Key': 63, 'ShortName': 'Sakhir'}},
 'ArchiveStatus': {'Status': 'Generating'},
 'Key': 6993,
 'Type': 'Race',
 'Name': 'Race',
 'StartDate': datetime.datetime(2022, 3, 20, 18, 0),
 'EndDate': datetime.datetime(2022, 3, 20, 20, 0),
 'GmtOffset': datetime.timedelta(seconds=10800),
 'Path': '2022/2022-03-20_Bahrain_Grand_Prix/2022-03-20_Race/'}

In [31]:
from queue import Queue


compd_q = Queue()
for idx in compounds_map.index:
    compd_q.put(compounds_map.loc[idx, :])

In [ ]:
comp_mapping_by_sess_key = {}
for s in sessions:
    while True:
        

In [15]:
sessions_by_circuit = processing.get_sessions_by_circuit(sessions)


In [ ]:

raw_data_by_circuit_split: Dict[str, List[pd.DataFrame]] = {}
for c, ss in sessions_by_circuit.items():
    raw_data_by_circuit_split[c] = []
    for s in ss:
        raw_data_by_circuit_split[c].append(processing.get_lap_data_with_weather(s))
        

In [23]:
# Raw data for each circuit
raw_data_by_circuit: Dict[str, pd.DataFrame] = {}
for c, dfs in raw_data_by_circuit_split.items():
    raw_data_by_circuit[c] = pd.concat(dfs, axis="index").reset_index()

In [24]:
# Clean data for each circuit
clean_data_for_each_circuit: Dict[str, pd.DataFrame] = {}
for c, df in raw_data_by_circuit.items():
    clean_data_for_each_circuit[c] = processing.get_refined_lap_data_with_z_score(df)

In [26]:
raw_data_by_circuit["Sakhir"]

,index,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,...,IsAccurate,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,LapTimeSeconds,LapTimeZScore
0,0,0 days 01:04:15.340000,VER,1,0 days 00:01:40.236000,1.0,1.0,NaT,NaT,NaT,...,False,23.9,26.0,1010.4,False,29.0,13,0.3,100.236,0.055495
1,1,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaT,NaT,0 days 00:00:31.285000,...,True,23.8,26.0,1010.4,False,29.0,357,0.5,97.880,-0.264151
2,2,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaT,NaT,0 days 00:00:31.499000,...,True,23.8,29.0,1010.2,False,28.8,13,0.4,98.357,-0.199435
3,3,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaT,NaT,0 days 00:00:31.342000,...,True,23.8,31.0,1010.4,False,28.7,50,0.3,98.566,-0.171079
4,4,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaT,NaT,0 days 00:00:31.498000,...,True,23.8,33.0,1010.4,False,28.5,316,0.4,98.877,-0.128885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4409,1109,0 days 02:25:33.298000,HUL,27,0 days 00:01:38.260000,53.0,3.0,NaT,NaT,0 days 00:00:31.255000,...,True,26.9,44.0,1008.4,False,30.4,8,1.5,98.260,-0.412171
4410,1110,0 days 02:27:11.796000,HUL,27,0 days 00:01:38.498000,54.0,3.0,NaT,NaT,0 days 00:00:31.388000,...,True,26.9,44.0,1008.4,False,30.5,33,1.9,98.498,-0.379881
4411,1111,0 days 02:28:50.220000,HUL,27,0 days 00:01:38.424000,55.0,3.0,NaT,NaT,0 days 00:00:31.415000,...,True,26.9,43.0,1008.4,False,30.3,44,1.2,98.424,-0.389921
4412,1112,0 days 02:30:29.218000,HUL,27,0 days 00:01:38.998000,56.0,3.0,NaT,NaT,0 days 00:00:31.369000,...,True,26.9,43.0,1008.4,False,30.4,44,1.4,98.998,-0.312044


In [ ]:
clean_data_for_each_circuit["Sakhir"]

,LapTimeZScore,IsPitLap,TyreLife,FreshTyre,LapNumber,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Compound_HARD,Compound_MEDIUM,Compound_SOFT
0,0.055495,False,4,False,1,23.9,26,1010.4,False,29.0,13,0.3,False,False,True
1,-0.264151,False,5,False,2,23.8,26,1010.4,False,29.0,357,0.5,False,False,True
2,-0.199435,False,6,False,3,23.8,29,1010.2,False,28.8,13,0.4,False,False,True
3,-0.171079,False,7,False,4,23.8,31,1010.4,False,28.7,50,0.3,False,False,True
4,-0.128885,False,8,False,5,23.8,33,1010.4,False,28.5,316,0.4,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4409,-0.412171,False,26,True,53,26.9,44,1008.4,False,30.4,8,1.5,False,True,False
4410,-0.379881,False,27,True,54,26.9,44,1008.4,False,30.5,33,1.9,False,True,False
4411,-0.389921,False,28,True,55,26.9,43,1008.4,False,30.3,44,1.2,False,True,False
4412,-0.312044,False,29,True,56,26.9,43,1008.4,False,30.4,44,1.4,False,True,False


In [ ]:
for c, d in raw_data_by_circuit_split.items():
    print(c, len(d))

Sakhir 4
Jeddah 4
Melbourne 4
Miami 2
Catalunya 4
Monte Carlo 4
Baku 3
Montreal 4
Silverstone 4
Paul Ricard 1
Hungaroring 4
Spa-Francorchamps 2
Zandvoort 4
Monza 4
Singapore 4
Suzuka 4
Austin 1
Mexico City 3
Yas Marina Circuit 3
Las Vegas 2
Imola 2
Spielberg 1


In [ ]:
sakhir_sessions